> **제1회 Level 모의고사** — 교재 591~631쪽
> 문제 591~593쪽 / 정답·해설 603~631쪽
>
> 노트북과 같은 위치에 데이터 파일(csv)을 두고 위에서부터 차례대로 실행하세요.

# 작업형 제1유형

## 문제 1. (10점)

제공된 데이터(coffee_consumption.csv)는 2000년부터 2023년까지 25개 국가의 연도별 1인당 커피 소비량(kg) 데이터입니다. 아래 소문제를 각각 구하시오.

**문제 1-1** 2000년부터 2023년까지 각 연도별로 커피 소비량이 가장 많은 국가를 찾고, 그 중 1위를 가장 많이 차지한 국가의 1위 횟수를 정수로 구하시오.

**문제 1-2** 다음 공식으로 각 국가의 CAGR(연평균 성장률)을 계산하고, 그 중 최댓값을 소수점 셋째 자리까지 구하시오.

```
CAGR = (EV / BV)^(1/n) - 1
```
- EV(Ending Value): 최종 값 (2023년 소비량)
- BV(Beginning Value): 초기 값 (2000년 소비량)
- n: 전체 기간 (2023년 - 2000년 = 23년)

**제공 데이터 (coffee_consumption.csv)**
| 컬럼명 | 설명 |
|---|---|
| Country | 국가명 (25개국) |
| 2000 ~ 2023 | 해당 연도의 1인당 커피 소비량(kg) |

**[제출 형식]**
- 문항 1-1: 정수(integer)로 작성
- 문항 1-2: 반올림하여 소수점 셋째 자리까지 작성

### 문제 풀이 방향

이 데이터는 국가가 행, 연도가 열로 되어 있는 형태입니다. 1-1은 "각 열(연도)에서 값이 가장 큰 행(국가)이 누구인지"를 24번 반복해서 찾은 뒤 그 결과를 집계하는 문제이고, 1-2는 국가별로 시작값과 끝값만 있으면 계산되는 공식이므로 컬럼 전체를 한 번에 계산하는 방식으로 풀 수 있습니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기 + 연도별 컬럼 목록 만들기**

In [1]:
import pandas as pd

df = pd.read_csv('coffee_consumption.csv')
year_cols = [str(y) for y in range(2000, 2024)]

print(df[['Country', '2000', '2001', '2023']].head())

   Country   2000   2001   2023
0  Finland  11.60  10.14  15.87
1   Norway  10.37  10.90  12.94
2   Sweden   4.09   4.87   7.92
3  Iceland   8.25   9.80  12.84
4  Denmark   5.00   4.32   5.32


데이터를 불러오고, 2000부터 2023까지의 연도 컬럼명을 문자열 리스트로 만듭니다. `pd.read_csv()`로 파일을 불러오면 헤더에 있던 값은 항상 문자열로 컬럼명이 되므로, 실제 컬럼명은 정수 2000이 아니라 문자열 `'2000'`입니다. 반면 `range(2000, 2024)`는 정수 2000, 2001, ...을 만들어냅니다. 이 둘의 타입이 다르기 때문에 변환 없이 `df[2000]`처럼 정수로 컬럼을 찾으려 하면 `KeyError`가 발생합니다. `str(y)`로 정수를 문자열로 바꿔야 실제 컬럼명과 타입이 일치하여 `df['2000']`처럼 정확히 접근할 수 있습니다.

> 💡 컬럼명이 특정 규칙(4자리 숫자 등)을 따를 때는 `df.filter(regex='^\\d{4}$')`처럼 정규표현식으로 조건에 맞는 컬럼만 자동으로 골라낼 수도 있습니다. 컬럼이 아주 많을 때 유용한 방법입니다.

**셀 2) 연도별 1위 국가 찾기**

In [2]:
top_country_by_year = df.set_index('Country')[year_cols].idxmax(axis=0)
print(top_country_by_year.head())

2000    Finland
2001     Norway
2002    Finland
2003    Finland
2004     Norway
dtype: object


`idxmax(axis=0)`은 각 컬럼(여기서는 각 연도)에서 값이 가장 큰 행의 인덱스를 반환합니다. `axis=0`은 각 컬럼을 위에서 아래 방향으로 훑으면서 최댓값의 위치를 찾는다는 뜻입니다. 이 문제에서는 값 자체(몇 kg인지)가 아니라 어느 국가인지를 알아야 하므로, 값을 반환하는 `max()`가 아니라 위치를 반환하는 `idxmax()`가 필요합니다.

`set_index('Country')`를 먼저 적용한 이유도 함께 보면 좋습니다. Country를 인덱스로 지정해두지 않으면 idxmax의 결과가 0, 1, 2 같은 행 번호로 나와서 어느 국가인지 알아보기 어렵습니다. 인덱스를 Country로 바꿔두면 idxmax가 국가 이름을 그대로 돌려주므로 결과를 바로 읽을 수 있습니다.

**셀 3) 최다 1위 국가의 횟수 구하기 (문제 1-1 정답)**

In [3]:
count_result = top_country_by_year.value_counts()
answer_1_1 = count_result.max()

print(count_result)
print("문제 1-1 정답:", answer_1_1)

Finland    17
Norway      5
Iceland     2
Name: count, dtype: int64
문제 1-1 정답: 17


`value_counts()`는 Series 안의 값들이 각각 몇 번씩 등장했는지 세어줍니다. 셀 2에서 만든 결과는 24개 연도 각각의 1위 국가 이름이 나열된 Series이므로, 여기에 value_counts를 적용하면 어느 국가가 몇 번 1위를 했는지가 바로 집계됩니다.

> 💡 `value_counts()`의 결과는 항상 등장 횟수가 많은 순서로 자동 정렬되어 있습니다. 그래서 `.max()`뿐 아니라 `.index[0]`으로 가장 많이 등장한 항목의 이름도 바로 꺼낼 수 있습니다. "가장 자주 나온 값이 무엇인가"를 묻는 문제에서 매번 유용하게 쓰이는 패턴입니다.

**셀 4) 국가별 CAGR 계산하기**

In [4]:
BV, EV, n = df['2000'], df['2023'], 23
df['CAGR'] = (EV / BV) ** (1/n) - 1

print(df[['Country', 'CAGR']].head())

   Country      CAGR
0  Finland  0.013720
1   Norway  0.009673
2   Sweden  0.029149
3  Iceland  0.019419
4  Denmark  0.002701


`BV`와 `EV`는 각각 2000년 컬럼과 2023년 컬럼 전체입니다. 컬럼끼리 나누면 같은 행끼리 자동으로 짝지어 계산되므로, 25개 국가의 CAGR을 반복문 없이 한 번에 구할 수 있습니다.

여기서 `n = 23`을 짚어볼 필요가 있습니다. 2000년부터 2023년까지 연도는 24개(2000, 2001, ..., 2023) 있지만, 성장이 일어나는 구간은 2000→2001, 2001→2002, ..., 2022→2023까지 총 23번입니다. 2000년 자체는 아직 성장이 시작되기 전의 출발점이므로 구간에 포함되지 않습니다. 쉽게 비교하면, 태어난 해를 1살로 세지 않고 만 나이를 계산할 때 생일이 몇 번 지났는지를 세는 것과 같은 원리입니다. CAGR의 n은 연도의 개수가 아니라 구간(간격)의 개수를 의미하므로 24가 아니라 23을 사용합니다.

**셀 5) CAGR 최댓값 찾기 (문제 1-2 정답)**

In [5]:
answer_1_2 = round(df['CAGR'].max(), 3)
print("문제 1-2 정답:", answer_1_2)

문제 1-2 정답: 0.075


CAGR 컬럼 전체에서 최댓값을 구한 뒤 마지막에만 반올림을 적용합니다. 중간 계산 단계에서 미리 반올림을 하면 그 값을 다음 계산에 사용하게 되어 오차가 조금씩 누적될 수 있으므로, 반올림은 항상 최종 결과를 출력하는 순간에만 적용하는 것이 안전합니다.

### 전체 코드

In [6]:
import pandas as pd

df = pd.read_csv('coffee_consumption.csv')
year_cols = [str(y) for y in range(2000, 2024)]

# 1-1: 연도별 1위 국가를 찾고, 가장 많이 1위를 한 횟수 구하기
top_country_by_year = df.set_index('Country')[year_cols].idxmax(axis=0)
answer_1_1 = top_country_by_year.value_counts().max()
print("문제 1-1 정답:", answer_1_1)

# 1-2: 국가별 CAGR을 계산하고 최댓값 구하기
BV, EV, n = df['2000'], df['2023'], 23
df['CAGR'] = (EV / BV) ** (1/n) - 1
answer_1_2 = round(df['CAGR'].max(), 3)
print("문제 1-2 정답:", answer_1_2)

문제 1-1 정답: 17
문제 1-2 정답: 0.075


### 정리
이 문제는 "국가별로 흩어진 값을 어떻게 하나의 답으로 모을 것인가"를 다루고 있습니다. 1-1은 idxmax로 위치를 찾고 value_counts로 빈도를 세는 흐름이었고, 1-2는 컬럼 두 개를 통째로 나누는 벡터 연산으로 25개 국가를 한꺼번에 계산했습니다. 데이터가 표 형태일 때는 어떤 축을 기준으로 계산할지(행 방향인지 열 방향인지)만 정확히 파악하면, 대부분 pandas가 제공하는 함수 한두 개로 반복 작업이 대체됩니다.

---

## 문제 2. (10점)

제공된 데이터(bookstore_order.csv)는 온라인 서점의 주문 데이터입니다. 전처리 후 아래 소문제를 각각 구하시오.

**[전처리 수행]**
- 새로운 컬럼 `order_amt` 생성: `order_amt = quantity × price`
- 새로운 컬럼 `cancel_TF` 생성: 주문번호(`order_no`)가 'C'로 시작하면 True, 아니면 False ('C'는 취소 주문)
- `cancel_TF`가 True인 행의 `order_amt`를 음수로 변환

**문제 2-1** 취소 주문 중 `order_amt`의 절댓값이 가장 큰 주문번호를 구하시오.

**문제 2-2** 고객ID(`cus_id`)별로 `order_amt`를 합산하여 가장 큰 값을 소수점 둘째 자리까지 구하시오.

**제공 데이터 (bookstore_order.csv)**
| 컬럼명 | 설명 |
|---|---|
| order_no | 주문번호 |
| cus_id | 고객 ID |
| product | 상품 카테고리 |
| quantity | 주문 수량 |
| price | 상품 단가(원) |

**[제출 형식]**
- 문항 2-1: 문자열(주문번호) 그대로 작성
- 문항 2-2: 반올림하여 소수점 둘째 자리까지 작성

### 문제 풀이 방향

전처리 지시사항을 순서대로 적용한 뒤에 두 소문제를 푸는 구조입니다. 먼저 주문금액을 계산하고, 주문번호로 취소 여부를 판별하고, 취소 주문의 금액에 부호를 씌우는 세 단계를 거칩니다. 이 전처리가 끝나면 2-1은 취소 주문만 따로 떼어내서 그 안에서 크기 비교를 하는 문제이고, 2-2는 고객이라는 기준으로 묶어서 합계를 비교하는 문제입니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기 + order_amt 생성**

In [7]:
import pandas as pd

df = pd.read_csv('bookstore_order.csv')
df['order_amt'] = df['quantity'] * df['price']

print(df[['order_no', 'quantity', 'price', 'order_amt']].head())

  order_no  quantity  price  order_amt
0    A0001         3   9800      29400
1    A0002         4  32000     128000
2    C0003         1  12000      12000
3    C0004         5  12000      60000
4    C0005         3  21000      63000


두 컬럼끼리 곱하면 같은 행끼리 자동으로 짝지어 계산되므로 반복문이 필요 없습니다. 엑셀에서 수식 하나를 전체 행에 채워 넣는 것과 비슷한 느낌입니다.

**셀 2) cancel_TF 생성 + order_amt 음수 변환**

In [8]:
df['cancel_TF'] = df['order_no'].str.startswith('C')
df.loc[df['cancel_TF'], 'order_amt'] = -df.loc[df['cancel_TF'], 'order_amt']

print(df[['order_no', 'cancel_TF', 'order_amt']].head())

  order_no  cancel_TF  order_amt
0    A0001      False      29400
1    A0002      False     128000
2    C0003       True     -12000
3    C0004       True     -60000
4    C0005       True     -63000


`.str.startswith('C')`는 문자열 컬럼 전체에 대해 첫 글자가 C인지를 한 번에 판별합니다. 이 데이터는 주문번호를 만들 때부터 정상 주문은 A로, 취소 주문은 C로 시작하도록 설계되어 있으므로 첫 글자만 확인하면 충분합니다. 학번 앞자리로 입학 연도를 표시하거나, 상품 코드 앞자리로 카테고리를 구분하는 것과 같은 방식입니다.

`df.loc[df['cancel_TF'], 'order_amt']`는 cancel_TF가 True인 행만 골라서 그 자리의 order_amt 값에 접근하는 방법입니다. 조건에 맞는 행만 선택해서 값을 바꿀 때는 `df.loc[조건, 컬럼]` 형태를 사용합니다.

취소 주문의 금액을 음수로 바꾸는 이유는 이후 계산을 단순하게 만들기 위해서입니다. 부호를 이렇게 미리 정해두면, 나중에 매출 합계를 구할 때 정상 주문과 취소 주문을 구분해서 따로 처리할 필요 없이 그냥 더하기만 하면 취소분이 자동으로 상쇄됩니다. 예를 들어 10만원을 주문했다가 3만원어치를 취소했다면 100000 + (-30000) = 70000처럼 한 번의 덧셈으로 실질 매출이 계산됩니다.

> 💡 `df.loc[조건, 컬럼] = 값`은 조건에 맞는 일부 행만 골라서 값을 바꾸는 대표적인 패턴입니다. 조건이 여러 개일 때는 `&`(and), `|`(or)로 연결할 수 있습니다. 예: `df.loc[(df['a']>0) & (df['b']=='X'), 'c'] = 0`

**셀 3) 취소 주문 중 절댓값이 가장 큰 주문번호 찾기 (문제 2-1 정답)**

In [9]:
cancel_df = df[df['cancel_TF']]
answer_2_1 = cancel_df.loc[cancel_df['order_amt'].abs().idxmax(), 'order_no']

print("문제 2-1 정답:", answer_2_1)

문제 2-1 정답: C0157


`df[df['cancel_TF']]`는 cancel_TF가 True인 행만 남긴 데이터프레임입니다.

여기서 `.abs()`가 왜 필요한지 짚어보겠습니다. 취소 주문의 order_amt는 모두 음수입니다(예: -12000, -60000, -128000). 절댓값 없이 그냥 order_amt가 가장 큰 값을 찾으면, 음수들 중 0에 가장 가까운 값(취소 금액이 가장 작은 것)이 가장 크다고 잘못 판단됩니다. 컴퓨터 입장에서는 -12000이 -128000보다 큰 숫자이기 때문입니다. 문제에서 요구하는 취소 규모가 가장 큰 주문은 부호를 무시한 크기가 가장 큰 값을 의미하므로, `.abs()`로 절댓값을 취한 뒤 `.idxmax()`로 그 위치를 찾아야 합니다.

**셀 4) 고객별 합산 최댓값 찾기 (문제 2-2 정답)**

In [10]:
answer_2_2 = round(df.groupby('cus_id')['order_amt'].sum().max(), 2)
print("문제 2-2 정답:", answer_2_2)

문제 2-2 정답: 714700


`groupby('cus_id')`는 cus_id 값이 같은 행들을 하나의 묶음으로 모읍니다. `['order_amt'].sum()`은 각 묶음 안에서 order_amt를 모두 더합니다. 셀 2에서 취소 주문을 음수로 바꿔두었기 때문에, 이 합계에는 정상 주문(+)과 취소 주문(-)이 자연스럽게 섞여 계산되어 고객별 실질 매출이 구해집니다. 마지막에 `.max()`를 붙이면 그 중 가장 큰 값이 나옵니다.

> 💡 `groupby(기준).집계함수()`는 1유형 전체에서 가장 많이 쓰이는 패턴입니다. 집계함수 자리에 `sum()`, `mean()`, `count()`, `max()`, `min()` 등을 바꿔 넣으면 원하는 통계를 그룹별로 바로 구할 수 있습니다.

### 전체 코드

In [11]:
import pandas as pd

df = pd.read_csv('bookstore_order.csv')

df['order_amt'] = df['quantity'] * df['price']
df['cancel_TF'] = df['order_no'].str.startswith('C')
df.loc[df['cancel_TF'], 'order_amt'] = -df.loc[df['cancel_TF'], 'order_amt']

# 2-1: 취소 주문 중 절댓값이 가장 큰 주문번호
cancel_df = df[df['cancel_TF']]
answer_2_1 = cancel_df.loc[cancel_df['order_amt'].abs().idxmax(), 'order_no']
print("문제 2-1 정답:", answer_2_1)

# 2-2: 고객별 합산 최댓값
answer_2_2 = round(df.groupby('cus_id')['order_amt'].sum().max(), 2)
print("문제 2-2 정답:", answer_2_2)

문제 2-1 정답: C0157
문제 2-2 정답: 714700


### 정리

이 문제의 핵심은 전처리 단계에서 미리 부호(음수)로 취소 여부를 표현해뒀다는 점입니다. 이렇게 데이터의 성격을 값 자체에 반영해두면, 이후의 집계(합계, 최댓값 찾기)가 조건문 없이 훨씬 단순해집니다. 다만 부호를 바꿔둔 데이터에서 크기를 비교할 때는 abs()로 부호를 무시해야 하는 순간과, 부호를 그대로 살려서 더해야 하는 순간(고객별 합산)을 구분할 수 있어야 합니다. 2-1은 전자, 2-2는 후자에 해당합니다.

---

## 문제 3. (10점)

제공된 데이터(employee_comp.csv)는 직원 130명의 부서, 성과등급, 근속연수, 연봉, 만족도 정보입니다. 아래 수행 순서에 따라 데이터를 처리한 후, 구한 값을 [제출 형식]에 맞춰 답안을 입력하시오.

**[수행 순서]**
1. 만족도(`satisfy`)의 결측치는 전체 만족도의 평균값으로 대체한다.
2. 근속연수(`year`)의 결측치는 같은 부서(`dept`) 내에서 동일한 성과등급(`grade`)을 가진 직원들의 평균 근속연수로 대체한다. 이때, 평균 근속연수는 소수점 이하를 절사하여 정수로 변환하여 사용한다.
3. 각 직원의 `연봉 / 근속연수`(a) 값을 계산하여, 그 값이 세 번째로 큰 직원의 근속연수를 구하시오.
4. 각 직원의 `연봉 / 만족도`(b) 값을 계산하여, 그 값이 두 번째로 큰 직원의 만족도를 구하시오.
5. (3)과 (4)의 값을 더한 결과를 구하시오.

**제공 데이터 (employee_comp.csv)**
| 컬럼명 | 설명 |
|---|---|
| id | 사번 |
| dept | 부서 |
| grade | 성과등급 (A/B/C) |
| year | 근속연수 |
| amount | 연봉(만원) |
| satisfy | 업무 만족도 (1~5점) |

**[제출 형식]**
- 반올림하여 소수점 둘째 자리까지 작성

### 문제 풀이 방향

이 문제는 결측치 처리 두 번, 파생변수 생성 두 번, 순위 조회 두 번, 마지막 합산까지 총 다섯 단계로 이어지는 복합 문제입니다. 단계가 많아 보이지만 하나씩 보면 결측치를 채우는 기준(전체 평균 vs 그룹 평균)이 다르다는 것과, 순위를 조회하는 방식이 동일하게 반복된다는 것만 파악하면 어렵지 않습니다. 수행 순서에 적힌 그대로 위에서부터 차례로 처리하면 됩니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기**

In [12]:
import pandas as pd

df = pd.read_csv('employee_comp.csv')
print(df.head())

     id dept grade  year  amount  satisfy
0  E001  마케팅     B   4.0    5460      NaN
1  E002   생산     B   4.0    4123      4.2
2  E003   인사     B  12.0    4456      1.6
3  E004   개발     A   7.0    5266      3.9
4  E005   생산     C   8.0    5143      2.9


**셀 2) 만족도 결측치를 전체 평균으로 대체**

In [13]:
df['satisfy'] = df['satisfy'].fillna(df['satisfy'].mean())
print(df['satisfy'].head())

0    2.957627
1    4.200000
2    1.600000
3    3.900000
4    2.900000
Name: satisfy, dtype: float64


`.mean()`은 결측치를 자동으로 제외하고 평균을 계산합니다. 결측치는 값이 없다는 의미이지 숫자 0이 아니므로, 평균 계산에 결측치가 그대로 포함되면 값이 왜곡됩니다. `fillna()`는 지정한 값으로 결측치를 채우는 함수이며, 여기서는 방금 구한 전체 평균을 그대로 넣었습니다. 문제에서 전체 평균으로 대체하라고 명시했으므로 조건 없이 데이터 전체를 대상으로 평균을 구한 것입니다.

**셀 3) 근속연수 결측치를 그룹 평균으로 대체 (소수점 버림)**

In [14]:
import numpy as np

group_mean = df.groupby(['dept', 'grade'])['year'].transform('mean')
df['year'] = df['year'].fillna(group_mean)
df['year'] = np.floor(df['year']).astype(int)

print(df['year'].head())

0     4
1     4
2    12
3     7
4     8
Name: year, dtype: int64


이번에는 전체 평균이 아니라 부서와 성과등급이 모두 같은 직원들끼리의 평균으로 채워야 합니다. `groupby(['dept', 'grade'])`처럼 두 개의 기준을 리스트로 함께 넘기면 두 조건을 모두 만족하는 행끼리 하나의 그룹으로 묶입니다. 조건이 dept 하나뿐이었다면 같은 부서 안에서 등급이 다른 직원들까지 한 그룹으로 묶여 문제가 요구하는 범위보다 넓어집니다.

`.transform('mean')`을 사용한 이유는 `fillna()`에 곧바로 넣을 수 있는 형태로 결과를 받기 위해서입니다. `groupby(['dept','grade'])['year'].mean()`만 사용하면 그룹별로 압축된 요약표(그룹 개수만큼의 행)가 나오는데, 이 요약표는 원본 데이터프레임과 행 개수가 다르기 때문에 곧바로 fillna에 넣을 수 없습니다. `transform()`은 그룹 평균을 계산하되 원본과 동일한 행 개수로 다시 펼쳐서 돌려주므로, 각 행에 자신이 속한 그룹의 평균이 그대로 대응되어 fillna에 바로 사용할 수 있습니다.

`np.floor()`를 사용한 이유는 문제에서 절사라는 표현을 썼기 때문입니다. 절사는 반올림하지 않고 소수점 이하를 무조건 잘라낸다는 뜻입니다. 평균이 9.8이어도 10이 아니라 9로 처리해야 합니다. `round()`는 소수점 값에 따라 올리거나 내릴 수 있어 이 요구사항에 맞지 않고, `np.floor()`는 항상 소수점 이하를 버리므로 절사의 의미와 정확히 일치합니다.

> 💡 groupby().transform()은 결측치를 그룹 기준으로 채울 때뿐 아니라, "그룹 평균과 개별 값의 차이"처럼 원본 행 단위로 그룹 통계를 사용해야 하는 모든 상황에서 유용하게 쓰이는 패턴입니다.

**셀 4) 파생변수 a, b 생성**

In [15]:
df['a'] = df['amount'] / df['year']
df['b'] = df['amount'] / df['satisfy']

print(df[['id', 'amount', 'year', 'a', 'satisfy', 'b']].head())

     id  amount  year            a   satisfy            b
0  E001    5460     4  1365.000000  2.957627  1846.074499
1  E002    4123     4  1030.750000  4.200000   981.666667
2  E003    4456    12   371.333333  1.600000  2785.000000
3  E004    5266     7   752.285714  3.900000  1350.256410
4  E005    5143     8   642.875000  2.900000  1773.448276


연봉을 근속연수로 나눈 a는 근속연수 1년당 받는 연봉을, 연봉을 만족도로 나눈 b는 만족도 1점당 받는 연봉을 의미합니다. 절대적인 연봉 크기만으로는 비교하기 어려운 두 직원을 비교할 때, 다른 값으로 나누어 비율로 바꾸면 상대적인 효율을 비교할 수 있습니다. 예를 들어 20년 근속에 연봉 6000만원인 직원과 1년 근속에 연봉 5000만원인 직원이 있다면, 절대 금액은 전자가 높지만 근속연수 대비 효율은 후자가 훨씬 높습니다.

**셀 5) a값 3번째로 큰 직원의 근속연수 (답3)**

In [16]:
answer3 = df.sort_values('a', ascending=False).iloc[2]['year']
print("답3:", answer3)

답3: 2


`sort_values('a', ascending=False)`는 a값을 기준으로 큰 값부터 작은 값 순서로 데이터프레임 전체를 재배열합니다. 정렬한 뒤에는 위에서부터 순서대로 1등, 2등, 3등이 되므로 `.iloc[0]`이 1등, `.iloc[1]`이 2등, `.iloc[2]`가 3등에 해당합니다. 인덱스 번호가 등수보다 1 작다는 점을 기억해두면 됩니다. `.iloc[2]`로 3등 행 전체를 가져온 뒤 `['year']`로 그 직원의 근속연수만 꺼냅니다.

> 💡 `sort_values(기준, ascending=False).iloc[n-1]`은 n등 값을 찾는 문제에서 그대로 재사용할 수 있는 패턴입니다. 1등이면 iloc[0], 5등이면 iloc[4]를 사용하면 됩니다.

**셀 6) b값 2번째로 큰 직원의 만족도 (답4)**

In [17]:
answer4 = df.sort_values('b', ascending=False).iloc[1]['satisfy']
print("답4:", answer4)

답4: 1.1


셀 5와 같은 방식이며, 이번에는 2등을 찾아야 하므로 `.iloc[1]`을 사용합니다.

**셀 7) 최종 정답 (답3 + 답4)**

In [18]:
final_answer = round(answer3 + answer4, 2)
print("최종 정답:", final_answer)

최종 정답: 3.1


### 전체 코드

In [19]:
import pandas as pd
import numpy as np

df = pd.read_csv('employee_comp.csv')

# 1) 만족도 결측치 -> 전체 평균
df['satisfy'] = df['satisfy'].fillna(df['satisfy'].mean())

# 2) 근속연수 결측치 -> (dept, grade) 그룹 평균, 절사하여 정수화
group_mean = df.groupby(['dept', 'grade'])['year'].transform('mean')
df['year'] = df['year'].fillna(group_mean)
df['year'] = np.floor(df['year']).astype(int)

# 3), 4) 파생변수 생성
df['a'] = df['amount'] / df['year']
df['b'] = df['amount'] / df['satisfy']

# 5) 순위 조회 및 합산
answer3 = df.sort_values('a', ascending=False).iloc[2]['year']
answer4 = df.sort_values('b', ascending=False).iloc[1]['satisfy']

final_answer = round(answer3 + answer4, 2)
print("답3:", answer3)
print("답4:", answer4)
print("최종 정답:", final_answer)

답3: 2
답4: 1.1
최종 정답: 3.1


### 정리

이 문제는 결측치 처리 방식이 컬럼마다 다르게 지시된다는 점이 핵심입니다. satisfy는 조건 없는 전체 평균으로, year는 (dept, grade)라는 두 가지 조건을 모두 만족하는 그룹 평균으로 채웠습니다. 이후 a, b라는 두 개의 비율 변수를 만들고 각각 정렬해서 원하는 등수의 값을 꺼내는 과정은 동일한 패턴(sort_values + iloc)의 반복이었습니다. 여러 단계가 이어지는 문제일수록, 각 단계에서 무엇을 기준으로 그룹을 나누는지와 최종적으로 어떤 컬럼의 값을 꺼내야 하는지를 문제 지문에서 정확히 확인하는 것이 중요합니다.

---

## 이번 회차에서 사용한 핵심 함수 정리
| 상황 | 함수 | 핵심 이유 |
|---|---|---|
| 컬럼(연도)별 최댓값의 위치(국가) 찾기 | `idxmax(axis=0)` | 값이 아니라 값의 위치(어느 행인지)가 필요할 때 사용 |
| 값별 등장 횟수 세기 | `value_counts()` | 결과가 자동으로 내림차순 정렬되어 순위 파악이 쉬움 |
| 두 컬럼 연산 | 컬럼끼리 직접 연산 (`+`, `-`, `*`, `/`) | 같은 행끼리 자동으로 짝지어 계산되어 반복문 불필요 |
| 문자열 조건 확인 | `str.startswith()` | 문자열 함수를 컬럼 전체에 한 번에 적용 |
| 조건에 맞는 값만 변경 | `df.loc[조건, 컬럼] = 값` | 조건을 만족하는 행만 정확히 선택하여 값 변경 |
| 결측치를 특정 값으로 채우기 | `fillna()` | 값이 없는 자리에만 정확히 채워짐 |
| 그룹별 평균을 원본 행 개수에 맞춰 계산 | `groupby().transform('mean')` | 그룹 요약표가 아니라 원본과 동일한 행 개수로 반환되어 fillna에 바로 사용 가능 |
| 소수점 버림(절사) | `np.floor()` | round와 달리 항상 아래로 버려서 절사의 의미와 일치 |
| n등 값 찾기 | `sort_values(ascending=False).iloc[n-1]` | 정렬 순서와 인덱스 번호를 이용해 등수를 바로 조회 |
| 그룹별 합계/평균 | `groupby().sum()` / `.mean()` | 조건별 집계를 한 줄로 처리 |

# 작업형 제2유형

## 문제

제공된 학습용 데이터(train.csv)는 자동차 보험 가입자의 개인정보 및 운행 특성 데이터이다. 학습용 데이터를 활용하여 가입자의 사고 위험등급(risk_level)을 예측하는 모형을 개발하고, 모형을 평가용 데이터(test.csv)에 적용하여 위험등급을 예측하시오. 예측 결과는 아래의 [유의사항]과 [제출 형식]을 준수하여, CSV 파일(result.csv)을 생성하는 코드를 제출하시오.

**[유의사항]**
- ① 예측 결과는 F1 Score(macro) 평가지표에 따라 평가함
- ② 타겟 변수(risk_level)는 0, 1, 2 총 3개의 클래스로 구성됨 (0: 저위험, 1: 중위험, 2: 고위험)

**[제출 형식]**
- ① CSV 파일명: result.csv
- ② 예측 위험등급 컬럼명: pred
- ③ 제출 컬럼 개수: pred 컬럼 1개
- ④ 평가용 데이터 개수와 예측 결과 데이터 개수 일치: 903개

**[제공 데이터]**
- train.csv: 학습용 데이터, 약 2,614개
- test.csv: 평가용 데이터, 약 903개 (risk_level 컬럼 미제공)

| 컬럼명 | 설명 |
|---|---|
| id | 가입자 ID |
| age | 나이 |
| gender | 성별 (M/F) |
| driving_experience | 운전 경력(년) |
| vehicle_age | 차량 연식(년) |
| vehicle_type | 차종 (세단/SUV/경차/트럭/스포츠카) |
| annual_mileage | 연간 주행거리(km) |
| past_accidents | 과거 사고 횟수 |
| credit_score | 신용점수 |
| region | 거주 지역 (서울/경기/지방) |
| marital_status | 결혼 여부 (기혼/미혼) |
| risk_level | (train에만 존재) 사고 위험등급 — 0: 저위험, 1: 중위험, 2: 고위험 |

### 문제 풀이 방향

2유형은 1유형과 달리 정답 하나를 구하는 것이 아니라, 학습 → 예측 → 제출파일 생성까지 이어지는 전체 파이프라인을 코드로 완성하는 것이 목표입니다. 순서는 다음과 같습니다.

1. 데이터를 불러와 전체적인 구조와 결측치의 여부를 확인한다
2. 결측치가 있다면 결측치를 처리한다
3. 문자로 된 범주형 변수를 숫자로 인코딩한다 (모델은 문자를 직접 이해하지 못하기 때문)
4. 학습 데이터 일부를 떼어내 검증용으로 사용하여, 실제 채점 전에 모델의 성능을 미리 가늠해본다
5. 모델을 학습시키고 검증 데이터로 F1-macro 점수를 확인한다
6. 확인이 끝나면 전체 학습 데이터로 다시 학습시킨 뒤, 진짜 평가 데이터(test.csv)를 예측한다
7. 요구된 형식(pred 컬럼 1개, 903행)으로 result.csv를 저장한다

### 정답 코드 및 해설

**셀 1) 데이터 불러오기 + 구조 확인**

In [1]:
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(train.shape, test.shape)
train.head()

(2614, 12) (903, 11)


,id,age,gender,driving_experience,vehicle_age,vehicle_type,annual_mileage,past_accidents,credit_score,region,marital_status,risk_level
0,365,45,F,27,19,경차,13440.0,1,617.0,지방,기혼,1
1,137,50,F,30,5,경차,14408.0,0,681.0,지방,기혼,0
2,100,59,F,36,10,트럭,11456.0,1,539.0,지방,기혼,0
3,443,25,M,6,6,세단,4382.0,0,817.0,지방,미혼,0
4,2741,19,F,0,14,경차,3265.0,0,574.0,지방,기혼,0


train은 12개 컬럼(risk_level 포함), test는 11개 컬럼(risk_level 없음)입니다. test에 risk_level이 없는 이유는, 이 값을 우리가 직접 예측해서 채워야 하기 때문입니다. train.shape과 test.shape을 먼저 확인하는 습관은 이후 제출 파일의 행 개수가 test와 정확히 일치하는지 확인할 때 기준이 됩니다.

**셀 2) 결측치 확인 및 처리**

In [2]:
print(train.isna().sum())
print(test.isna().sum())
# train.info()

id                     0
age                    0
gender                 0
driving_experience     0
vehicle_age            0
vehicle_type           0
annual_mileage        65
past_accidents         0
credit_score          95
region                 0
marital_status         0
risk_level             0
dtype: int64
id                     0
age                    0
gender                 0
driving_experience     0
vehicle_age            0
vehicle_type           0
annual_mileage        25
past_accidents         0
credit_score          25
region                 0
marital_status         0
dtype: int64


isna로 결측치의 여부를 확인하고, 이를 sum으로 합산하면 결측치의 개수를 확인할 수 있습니다. annual_mileage와 credit_score에 결측치가 있습니다.

결측치의 여부는 info()를 이용해 확인할 수도 있습니다.

In [3]:
for col in ['annual_mileage', 'credit_score']:
    med = train[col].median()
    train[col] = train[col].fillna(med)
    test[col] = test[col].fillna(med)

print(train.isna().sum().sum(), test.isna().sum().sum())

0 0


여기서는 결측치를 중앙값으로 채웠습니다. 평균 대신 중앙값을 쓴 이유는 연간 주행거리나 신용점수처럼 극단적으로 크거나 작은 값(이상치)이 섞여 있을 수 있는 컬럼에서는 평균이 그 이상치에 영향을 크게 받는 반면 중앙값은 상대적으로 안정적이기 때문입니다.

한 가지 중요한 점은, 중앙값을 train 데이터에서만 구해서 train과 test 양쪽에 동일하게 적용했다는 것입니다. test 데이터의 중앙값을 따로 구해서 채우면 안 됩니다. 실제 상황에서는 test 데이터가 어떤 값을 가지고 있는지 미리 알 수 없다고 가정해야 하므로, 결측치를 채우는 기준은 항상 train에서만 만들어서 두 데이터에 동일하게 적용해야 합니다.

> 💡 이렇게 train에서 계산한 기준값(중앙값, 평균, 인코딩 규칙 등)을 test에 그대로 적용하는 방식은 2유형 전체에서 반복되는 원칙입니다. train과 test를 각각 따로 처리하면 두 데이터의 기준이 어긋나서 예측이 왜곡될 수 있습니다.

**셀 3) 범주형 변수 인코딩**

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2614 entries, 0 to 2613
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  2614 non-null   int64  
 1   age                 2614 non-null   int64  
 2   gender              2614 non-null   object 
 3   driving_experience  2614 non-null   int64  
 4   vehicle_age         2614 non-null   int64  
 5   vehicle_type        2614 non-null   object 
 6   annual_mileage      2614 non-null   float64
 7   past_accidents      2614 non-null   int64  
 8   credit_score        2614 non-null   float64
 9   region              2614 non-null   object 
 10  marital_status      2614 non-null   object 
 11  risk_level          2614 non-null   int64  
dtypes: float64(2), int64(6), object(4)
memory usage: 245.2+ KB


gender, vehicle_type, region, marital_status는 문자열로 되어 있는데(info 결과의 Dtype에서 확인), 대부분의 머신러닝 모델은 숫자만 입력으로 받을 수 있습니다.

In [5]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['gender', 'vehicle_type', 'region', 'marital_status']

for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([train[col], test[col]]))
    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])

print(train[cat_cols].head())

   gender  vehicle_type  region  marital_status
0       0             1       2               0
1       0             1       2               0
2       0             4       2               0
3       1             2       2               1
4       0             1       2               0


 `LabelEncoder`는 문자열 범주를 0, 1, 2, ...라는 정수로 바꿔주는 도구입니다.

여기서 `le.fit(pd.concat([train[col], test[col]]))`처럼 train과 test를 합친 데이터로 fit을 했다는 점이 중요합니다. 만약 train 데이터만으로 fit을 했는데 test 데이터에 train에는 없던 범주값이 등장한다면, transform 과정에서 에러가 발생합니다. train과 test를 합쳐서 fit하면 두 데이터에 등장하는 모든 범주를 미리 인코더가 알고 있게 되어 이런 문제를 방지할 수 있습니다.

**셀 4) 학습용/검증용 데이터 분리**

In [6]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=['id', 'risk_level'])
y = train['risk_level']
X_test = test.drop(columns=['id'])

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_tr.shape, X_val.shape)

(2091, 10) (523, 10)


`id`는 각 행을 구분하기 위한 값일 뿐 위험등급과는 아무 관련이 없으므로 학습에 사용할 컬럼(X)에서 제외합니다. `y`는 우리가 맞추고 싶은 정답(risk_level)입니다.

`train_test_split`은 train 데이터를 다시 두 조각으로 나눕니다. 이렇게 나누는 이유는, 진짜 test.csv에는 정답이 없어서 우리가 만든 모델이 얼마나 잘 맞추는지 미리 확인할 방법이 없기 때문입니다. train 데이터 중 일부(여기서는 20%)를 검증용으로 따로 떼어두면, 그 부분에 대해서는 우리가 정답(y_val)을 알고 있으므로 모델의 예측(val_pred)과 비교해서 점수를 미리 가늠해볼 수 있습니다.

`stratify=y`라는 옵션도 짚어볼 필요가 있습니다. risk_level은 0, 1, 2의 비율이 원래 데이터에서 고르지 않습니다(저위험이 가장 많고 고위험이 가장 적음). 만약 stratify 없이 무작위로 나누면 우연히 검증용 데이터에 특정 등급이 거의 없거나 지나치게 몰릴 수 있습니다. `stratify=y`는 나눌 때 원본 데이터의 클래스 비율을 학습용과 검증용 양쪽에 최대한 동일하게 유지해주는 옵션입니다.

**셀 5) 모델 학습 및 검증 성능 확인**

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_tr, y_tr)

val_pred = model.predict(X_val)
print("검증 F1-macro:", f1_score(y_val, val_pred, average='macro'))

검증 F1-macro: 0.6744584992683692


`RandomForestClassifier`는 여러 개의 결정나무(의사결정 트리)를 만들고 그 예측을 종합해서 최종 판단을 내리는 모델입니다. 특별한 전처리(스케일링 등)가 크게 필요하지 않고 안정적인 성능을 내는 경우가 많아 2유형에서 기본으로 시도하기 좋은 모델입니다.

`f1_score(..., average='macro')`는 문제에서 요구한 평가지표와 동일한 방식으로 점수를 계산합니다. macro 방식은 클래스별로(0, 1, 2 각각) F1 점수를 따로 구한 뒤 그 세 점수를 단순 평균 냅니다. 클래스 개수가 적은 고위험군을 잘 맞추는지도 저위험군만큼 동등하게 중요하게 평가한다는 의미입니다. 만약 데이터가 많은 클래스(저위험)만 잘 맞추고 데이터가 적은 클래스(고위험)를 거의 못 맞추는 모델이라면, macro F1은 이를 낮은 점수로 반영합니다.

> 💡 F1-macro가 요구되는 문제에서는 모델 학습 시 `class_weight='balanced'` 옵션을 시도해보는 것도 좋은 전략입니다. 데이터가 적은 클래스에 더 큰 가중치를 부여해서 소수 클래스를 더 잘 맞추도록 유도할 수 있습니다.

**셀 6) 전체 학습 데이터로 재학습 후 test 예측**

In [8]:
model.fit(X, y)
pred = model.predict(X_test)

print(pred[:10])

[1 2 0 0 0 2 1 1 0 1]


셀 5에서는 검증을 위해 train 데이터의 80%만 사용해서 학습했습니다. 성능을 확인했으니, 이제는 가지고 있는 train 데이터를 하나도 남기지 않고 전부 사용해서 다시 학습시킵니다. 데이터를 더 많이 사용할수록 일반적으로 모델이 더 안정적으로 패턴을 학습하기 때문입니다. 이렇게 재학습한 모델로 진짜 평가 대상인 test 데이터를 예측합니다.

**셀 7) 제출 파일 생성**

In [9]:
result = pd.DataFrame({'pred': pred})
result.to_csv('result.csv', index=False)

print(result.shape)
print(result.head())

(903, 1)
   pred
0     1
1     2
2     0
3     0
4     0


제출 형식에서 요구한 대로 `pred`라는 컬럼 하나만 있는 데이터프레임을 만들고, `to_csv()`로 저장합니다. `index=False`를 반드시 넣어야 합니다. 이 옵션이 없으면 행 번호(0, 1, 2, ...)가 별도의 컬럼으로 파일에 함께 저장되어, 요구된 pred 컬럼 1개라는 조건을 어기게 됩니다. 저장 후에는 `result.shape`으로 행 개수가 test 데이터 개수(903개)와 정확히 일치하는지 반드시 확인해야 합니다.

### 전체 코드

In [10]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 결측치 처리 (train 기준으로 계산한 값을 test에도 동일 적용)
for col in ['annual_mileage', 'credit_score']:
    med = train[col].median()
    train[col] = train[col].fillna(med)
    test[col] = test[col].fillna(med)

# 범주형 인코딩
cat_cols = ['gender', 'vehicle_type', 'region', 'marital_status']
for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([train[col], test[col]]))
    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])

X = train.drop(columns=['id', 'risk_level'])
y = train['risk_level']
X_test = test.drop(columns=['id'])

# 검증용 분리 후 성능 확인
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_tr, y_tr)
val_pred = model.predict(X_val)
print("검증 F1-macro:", f1_score(y_val, val_pred, average='macro'))

# 전체 데이터로 재학습 후 test 예측
model.fit(X, y)
pred = model.predict(X_test)

# 제출 파일 생성
result = pd.DataFrame({'pred': pred})
result.to_csv('result.csv', index=False)
print(result.shape)

검증 F1-macro: 0.6744584992683692
(903, 1)


### 정리

이 문제는 결측치 처리, 범주형 인코딩, 검증 데이터 분리, 모델 학습, 예측, 제출 파일 생성이라는 2유형의 전형적인 흐름을 담고 있습니다. 특히 강조하고 싶은 것은 train에서 계산한 기준(중앙값, 인코딩 규칙)을 test에 그대로 적용해야 한다는 원칙과, 클래스 비율이 불균형할 때 stratify로 분리하고 F1-score로 평가해야 하는 이유입니다. 실전에서는 이 기본 파이프라인을 완성한 뒤, 시간이 남으면 다른 모델(XGBoost, LightGBM 등 시험 환경에 설치되어 있다면)을 시도해보거나 `class_weight='balanced'` 같은 옵션을 바꿔가며 검증 점수를 비교해보는 것도 좋은 전략입니다.

이번 모델의 검증 F1-macro는 약 0.67~0.70 수준이었습니다. 채점은 실제 test.csv의 정답과 비교해 이루어지므로, 검증 점수와 실제 채점 점수가 정확히 같지는 않지만 비슷한 범위로 나오는 것이 일반적입니다.

---

## 이번 회차에서 사용한 핵심 개념 정리
| 상황 | 함수/개념 | 핵심 이유 |
|---|---|---|
| 문자열 범주를 숫자로 변환 | `LabelEncoder` | 대부분의 모델은 숫자 입력만 처리 가능 |
| train/test를 합쳐서 인코더 학습 | `le.fit(pd.concat([train[c], test[c]]))` | test에만 있는 미지의 범주로 인한 에러 방지 |
| 결측치 대체 기준값은 train에서만 계산 | `train[col].median()`을 test에도 적용 | test 정보를 미리 알고 있다고 가정하면 안 됨(데이터 누수 방지) |
| 정답을 모르는 test 이전에 성능 가늠하기 | `train_test_split` | train 일부를 검증용으로 떼어 미리 점수 확인 |
| 클래스 불균형 상태로 데이터 분리 | `stratify=y` | 학습용/검증용에 클래스 비율을 동일하게 유지 |
| 다중분류 평가 | `f1_score(..., average='macro')` | 클래스별 F1을 동일 가중치로 평균 (소수 클래스도 동등하게 반영) |
| 제출 파일 저장 | `to_csv('result.csv', index=False)` | index=False 없으면 불필요한 인덱스 컬럼이 추가됨 |

---

# 작업형 제3유형

## 문제 1. (10점)

제공된 데이터(hospital_readmission.csv)는 병원에서 수집한 퇴원 환자의 건강 특성 데이터이다. 나이, 만성질환 여부, 운동 빈도, 공복 혈당 등에 따른 재입원 여부(readmission)를 예측하기 위해 로지스틱 회귀분석(Logistic Regression)을 수행하고, 아래 소문항에 답하시오.

**제공 데이터 (hospital_readmission.csv)**
| 컬럼명 | 설명 |
|---|---|
| id | 환자 번호 |
| age | 나이 |
| chronic_disease | 만성질환 이력 (no/yes) |
| exercise_days_per_week | 주간 운동 일수 |
| avg_glucose | 평균 공복 혈당 |
| bmi | 체질량지수 |
| blood_pressure | 혈압 |
| readmission | (종속변수) 재입원 여부 — 0: 재입원 안함, 1: 재입원 |

**문제 1-1** id를 제외한 모든 변수를 독립변수로 사용하여 로지스틱 회귀모형을 적합하시오. chronic_disease가 'no'인 사람 대비 'yes'인 사람의 오즈비(Odds Ratio)를 구하시오. (반올림하여 소수 둘째 자리까지 작성)

**문제 1-2** 위 모형에서 유의확률(p-value)이 0.05보다 작은 변수만 사용하여 로지스틱 회귀모형을 다시 적합하시오. 이 모형의 회귀계수(절편 포함)의 합을 구하시오. (반올림하여 소수 셋째 자리까지 작성)

**문제 1-3** 1-2에서 적합한 모형을 이용하여, 새로운 환자(age=65, chronic_disease=yes, exercise_days_per_week=1, avg_glucose=160)의 재입원 확률을 예측하시오. (반올림하여 소수 셋째 자리까지 작성)

### 문제 풀이 방향

이 문제는 "전체 변수로 모형 적합 → 유의한 변수만 골라 재적합 → 그 모형으로 해석 및 예측"이라는 3유형의 전형적인 흐름을 담고 있습니다. 로지스틱 회귀는 sklearn이 아니라 `statsmodels`로 적합하는 것이 일반적인데, 그 이유는 sklearn의 로지스틱 회귀는 예측 성능에 초점이 맞춰져 있어 p-value나 신뢰구간 같은 통계적 검정 결과를 기본적으로 제공하지 않는 반면, statsmodels는 회귀계수의 유의성을 바로 확인할 수 있는 `summary()` 결과를 제공하기 때문입니다. 3유형처럼 "어떤 변수가 유의한가", "그 계수는 얼마인가"를 묻는 문제에는 statsmodels가 훨씬 적합합니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기**


In [1]:
import pandas as pd

df = pd.read_csv('hospital_readmission.csv')
print(df.shape)
df.head()

(850, 8)


,id,age,chronic_disease,exercise_days_per_week,avg_glucose,bmi,blood_pressure,readmission
0,1,62,yes,4,173.9,33.8,109.4,1
1,2,38,no,3,109.9,26.9,118.1,0
2,3,33,no,2,126.9,18.2,160.1,0
3,4,84,no,7,98.0,26.4,137.1,0
4,5,33,no,3,120.4,23.5,133.0,0


850명의 환자 데이터이며, 문제에서 언급되었던 chronic_disease는 문자열('no'/'yes')로 되어 있는 것을 확인할 수 있습니다. readmission이 우리가 예측하고자 하는 종속변수(타겟)입니다.

**셀 2) 범주형 변수를 숫자로 변환**

In [3]:
print(df['chronic_disease'].value_counts())

df['chronic_disease'] = df['chronic_disease'].map({'no': 0, 'yes': 1})
print(df[['chronic_disease']].head())

chronic_disease
no     508
yes    342
Name: count, dtype: int64
   chronic_disease
0                1
1                0
2                0
3                0
4                0


회귀모형은 숫자만 입력받을 수 있으므로, 'no'/'yes'라는 문자열을 0/1로 바꿔야 합니다. `.map({'no': 0, 'yes': 1})`은 딕셔너리에 지정한 대응 관계대로 값을 치환해주는 함수입니다. 여기서 no를 0으로, yes를 1로 지정했다는 것이 중요합니다. 이후에 구할 오즈비는 "기준값(0, 즉 no) 대비 비교값(1, 즉 yes)의 위험도가 몇 배인가"를 의미하는데, 이 기준을 어떻게 잡았는지가 오즈비 해석의 방향을 결정하기 때문입니다. 문제에서 "no인 사람 대비 yes인 사람의 오즈비"를 물었으므로, no=0, yes=1로 매핑한 것이 정확히 이 요구와 일치합니다.

**셀 3) 독립변수(X)와 종속변수(y) 구성 + 절편 추가**

In [4]:
import statsmodels.api as sm

X = df.drop(columns=['id', 'readmission'])
y = df['readmission']
X = sm.add_constant(X)

print(X.head())

   const  age  chronic_disease  exercise_days_per_week  avg_glucose   bmi  \
0    1.0   62                1                       4        173.9  33.8   
1    1.0   38                0                       3        109.9  26.9   
2    1.0   33                0                       2        126.9  18.2   
3    1.0   84                0                       7         98.0  26.4   
4    1.0   33                0                       3        120.4  23.5   

   blood_pressure  
0           109.4  
1           118.1  
2           160.1  
3           137.1  
4           133.0  


id는 환자를 구분하는 번호일 뿐 재입원 여부와 관련이 없으므로 독립변수에서 제외했습니다.

`sm.add_constant(X)`가 이번 셀에서 가장 낯설 수 있는 부분입니다. 회귀식은 원래 `y = b0 + b1*x1 + b2*x2 + ...` 형태로, b0(절편, 상수항)이 포함됩니다. sklearn의 회귀 모델은 절편을 내부적으로 자동으로 계산해주지만, statsmodels는 그렇지 않습니다. statsmodels에게 절편을 계산하라고 명시적으로 알려주려면, X 데이터에 값이 모두 1인 `const`라는 컬럼을 직접 추가해야 합니다. `sm.add_constant(X)`가 바로 이 작업을 자동으로 해줍니다. 만약 이 과정을 빠뜨리면, 모형은 원점(0)을 반드시 지나야 한다는 잘못된 가정하에 적합되어 절편이 아예 없는 모형이 되어버립니다.

**셀 4) 로지스틱 회귀모형 적합**

In [5]:
model = sm.Logit(y, X).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.562496
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:            readmission   No. Observations:                  850
Model:                          Logit   Df Residuals:                      843
Method:                           MLE   Df Model:                            6
Date:                Thu, 09 Jul 2026   Pseudo R-squ.:                  0.1435
Time:                        11:12:33   Log-Likelihood:                -478.12
converged:                       True   LL-Null:                       -558.22
Covariance Type:            nonrobust   LLR p-value:                 5.388e-32
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                     -5.1700      0.983     -5.260      0.000      -7.096      

`sm.Logit(y, X)`는 로지스틱 회귀모형 객체를 만들고, `.fit()`을 붙여야 실제로 데이터에 적합됩니다. `.fit()`을 빠뜨리면 계수가 계산되지 않은 빈 모형만 만들어지므로 주의해야 합니다.

`summary()` 표에서 눈여겨봐야 할 두 컬럼은 `coef`(회귀계수)와 `P>|z|`(p-value)입니다. `coef` 열을 보면 chronic_disease의 계수가 0.8060이라는 것을 알 수 있고, `P>|z|` 열을 보면 age, chronic_disease, exercise_days_per_week, avg_glucose는 값이 매우 작아(0.000으로 표시) 통계적으로 유의한 반면, bmi(0.317)와 blood_pressure(0.540)는 0.05보다 훨씬 커서 유의하지 않다는 것을 알 수 있습니다. 즉 이 두 변수는 재입원 여부를 설명하는 데 통계적으로 의미 있는 기여를 하지 못한다고 판단할 수 있습니다.

> 💡 시험에서는 `summary()`의 표 전체를 눈으로 훑기보다, `model.params`(계수만)와 `model.pvalues`(p-value만)를 따로 뽑아서 코드로 조건을 걸어 처리하는 것이 훨씬 정확하고 빠릅니다. 다음 셀에서 이 방법을 사용합니다.

**셀 5) 오즈비 계산 (문제 1-1 정답)**

In [6]:
import numpy as np

odds_ratio = np.exp(model.params['chronic_disease'])
print("1-1 정답:", round(odds_ratio, 2))

1-1 정답: 2.24


로지스틱 회귀에서 각 변수의 계수(coef)는 그 자체로는 직관적으로 해석하기 어렵습니다. 계수는 "그 변수가 1 증가할 때 로그 오즈(log-odds)가 얼마나 증가하는가"를 나타내는데, 로그 스케일이라 바로 와닿지 않기 때문입니다. 이 계수에 지수함수(`np.exp()`)를 취하면 로그 오즈가 오즈비(Odds Ratio)로 바뀌면서 "몇 배"라는 훨씬 이해하기 쉬운 해석이 가능해집니다.

chronic_disease의 계수는 0.8060이었고, 여기에 지수를 취한 값이 약 2.24입니다. 이는 다른 조건이 모두 같을 때, 만성질환이 없는 사람(no=0) 대비 있는 사람(yes=1)의 재입원 오즈(odds, 즉 재입원할 가능성과 안 할 가능성의 비)가 약 2.24배 높다는 뜻입니다. 셀 2에서 no를 0, yes를 1로 매핑해뒀기 때문에, 이 오즈비는 자연스럽게 "no 대비 yes"라는 문제의 요구 방향과 일치합니다.

**셀 6) 유의한 변수 선별**

In [7]:
sig_vars = model.pvalues[model.pvalues < 0.05].index.tolist()
print("유의한 변수(절편 포함):", sig_vars)

유의한 변수(절편 포함): ['const', 'age', 'chronic_disease', 'exercise_days_per_week', 'avg_glucose']


`model.pvalues`는 각 변수 이름을 인덱스로, p-value를 값으로 가지는 Series입니다. `model.pvalues < 0.05`는 각 변수의 p-value가 0.05보다 작은지를 True/False로 판별하고, 이 조건으로 다시 `model.pvalues[...]`를 감싸면 조건을 만족하는 행만 걸러집니다. `.index.tolist()`로 그 변수들의 이름만 리스트로 뽑았습니다. 결과를 보면 const(절편), age, chronic_disease, exercise_days_per_week, avg_glucose 5개가 유의한 변수로 남았고, bmi와 blood_pressure는 제외되었습니다. const가 항상 포함된 것은, 절편의 p-value(0.000)도 계산 대상이 되어 자동으로 조건을 만족했기 때문입니다.

**셀 7) 유의한 변수만으로 재적합**

In [8]:
X2 = X[sig_vars]
model2 = sm.Logit(y, X2).fit()
print(model2.summary())

Optimization terminated successfully.
         Current function value: 0.563311
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:            readmission   No. Observations:                  850
Model:                          Logit   Df Residuals:                      845
Method:                           MLE   Df Model:                            4
Date:                Thu, 09 Jul 2026   Pseudo R-squ.:                  0.1422
Time:                        11:14:08   Log-Likelihood:                -478.81
converged:                       True   LL-Null:                       -558.22
Covariance Type:            nonrobust   LLR p-value:                 2.634e-33
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                     -4.2332      0.463     -9.148      0.000      -5.140      

`X[sig_vars]`로 셀 6에서 찾은 5개 컬럼만 골라낸 뒤, 다시 `sm.Logit(y, X2).fit()`으로 새 모형을 적합했습니다. 새 모형에서는 남아있는 변수 4개(const 제외)가 모두 p-value 0.000으로 뚜렷하게 유의합니다. bmi와 blood_pressure라는 잡음에 가까운 변수를 제거하고 나니 남은 변수들의 유의성이 더 선명해진 것을 확인할 수 있습니다.

**셀 8) 유의한 계수의 합 (문제 1-2 정답)**

In [9]:
coef_sum = round(model2.params.sum(), 3)
print("1-2 정답:", coef_sum)

1-2 정답: -3.582


`model2.params`는 재적합한 모형의 회귀계수들을 담은 Series이며, `.sum()`으로 이 값들을 전부 더했습니다. 문제에서 "절편 포함"이라고 명시했으므로, const 계수(-4.2332)도 포함해서 합산해야 합니다. 만약 절편을 제외하고 싶다면 `model2.params.drop('const').sum()`처럼 drop으로 먼저 제거한 뒤 합산하면 됩니다. 이런 사소해 보이는 "절편 포함/제외" 지시사항을 놓치면 답이 완전히 달라지므로, 문제 지문을 꼼꼼히 읽는 것이 중요합니다.

**셀 9) 새로운 환자의 재입원 확률 예측 (문제 1-3 정답)**

In [10]:
new_data = pd.DataFrame({
    'const': [1], 'age': [65], 'chronic_disease': [1], 'exercise_days_per_week': [1],
    'avg_glucose': [160]
})
print(new_data)

pred_prob = model2.predict(new_data)
print("1-3 정답:", round(pred_prob.values[0], 3))

   const  age  chronic_disease  exercise_days_per_week  avg_glucose
0      1   65                1                       1          160
1-3 정답: 0.765


새로운 환자의 정보를 데이터프레임으로 만들 때, 컬럼 이름과 순서를 모형을 적합할 때 사용한 X2와 정확히 맞춰야 합니다. `const` 컬럼도 반드시 포함하고 값은 1로 넣어야 하는데, 이는 절편을 계산에 포함시키기 위한 것으로 셀 3에서 `sm.add_constant()`로 학습 데이터에 넣어줬던 것과 같은 이유입니다. bmi와 blood_pressure는 모형에서 이미 제거되었으므로 새 데이터에도 포함시키지 않았습니다. 만약 여기에 불필요한 컬럼을 넣거나 순서를 다르게 하면 `predict()`가 에러를 내거나 엉뚱한 값을 계산할 수 있습니다.

`model.predict()`는 로지스틱 회귀에서 별도의 옵션 없이 사용하면 곧바로 확률값(0~1 사이)을 돌려줍니다. 이는 2유형에서 sklearn의 `predict_proba()`가 클래스별 확률 두 개를 표 형태로 돌려주던 것과 다른 점입니다. statsmodels의 `predict()`는 애초에 "양성 클래스(1)일 확률" 하나만 돌려주도록 설계되어 있습니다.

### 전체 코드

In [11]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_csv('hospital_readmission.csv')
df['chronic_disease'] = df['chronic_disease'].map({'no': 0, 'yes': 1})

X = df.drop(columns=['id', 'readmission'])
y = df['readmission']
X = sm.add_constant(X)

# 전체 변수로 적합
model = sm.Logit(y, X).fit()

# 1-1: 오즈비
odds_ratio = round(np.exp(model.params['chronic_disease']), 2)
print("1-1 정답:", odds_ratio)

# 유의한 변수만 재적합
sig_vars = model.pvalues[model.pvalues < 0.05].index.tolist()
X2 = X[sig_vars]
model2 = sm.Logit(y, X2).fit()

# 1-2: 유의 계수 합(절편 포함)
coef_sum = round(model2.params.sum(), 3)
print("1-2 정답:", coef_sum)

# 1-3: 새 데이터 예측
new_data = pd.DataFrame({
    'const': [1], 'age': [65], 'chronic_disease': [1], 'exercise_days_per_week': [1],
    'avg_glucose': [160]
})
pred_prob = round(model2.predict(new_data).values[0], 3)
print("1-3 정답:", pred_prob)

Optimization terminated successfully.
         Current function value: 0.562496
         Iterations 6
1-1 정답: 2.24
Optimization terminated successfully.
         Current function value: 0.563311
         Iterations 6
1-2 정답: -3.582
1-3 정답: 0.765


### 정리

이 문제는 로지스틱 회귀에서 반드시 알아야 할 세 가지를 모두 다뤘습니다. `sm.add_constant()`로 절편을 직접 추가해야 한다는 것, 계수에 `np.exp()`를 취하면 오즈비로 해석할 수 있다는 것, 그리고 p-value로 유의한 변수만 골라 더 단순한 모형으로 재적합하는 흐름입니다. 특히 오즈비를 구할 때는 범주형 변수를 어떤 값으로 매핑했는지(0=no, 1=yes)에 따라 "무엇 대비 무엇"이라는 해석의 방향이 결정된다는 점을 기억해두면 좋습니다.

---

## 문제 2. (10점)

제공된 데이터(crop_yield.csv)는 농작물 재배 환경과 수확량에 관한 자료이다. 강수량, 평균 기온, 비료 사용량, 일조 시간 등에 따른 수확량(crop_yield)을 예측하기 위해 다중선형 회귀분석을 수행하고, 아래 소문항에 답하시오.

**제공 데이터 (crop_yield.csv)**
| 컬럼명 | 설명 |
|---|---|
| id | 재배 구역 번호 |
| rainfall | 강수량(mm) |
| avg_temp | 평균 기온(℃) |
| fertilizer_amount | 비료 사용량(kg) |
| sunlight_hours | 일조 시간(시간/일) |
| crop_yield | (종속변수) 수확량(kg/ha) |

**문제 2-1** id를 제외한 모든 변수를 독립변수로 사용하여 다중선형 회귀모형을 적합하시오. 절편을 제외한 회귀계수의 합을 구하시오. (반올림하여 소수 셋째 자리까지 작성)

**문제 2-2** 위 모형에서 유의확률(p-value)이 0.05보다 작은 변수만 사용하여 다중선형 회귀모형을 다시 적합하시오. 이 모형의 결정계수(R-squared) 값을 구하시오. (반올림하여 소수 셋째 자리까지 작성)

**문제 2-3** 2-2에서 적합한 모형을 이용하여, 새로운 재배 구역(rainfall=1000, avg_temp=24, fertilizer_amount=200)의 수확량을 예측하시오. (반올림하여 소수 셋째 자리까지 작성)

### 문제 풀이 방향

이 문제는 문제 1과 거의 같은 흐름(전체 변수로 적합 → 유의한 변수만 재적합 → 해석 및 예측)을 다중선형 회귀에 적용합니다. 다만 종속변수(crop_yield)가 0/1이 아니라 연속적인 수치이므로 `sm.Logit` 대신 `sm.OLS`를 사용하고, 로지스틱 회귀에는 없던 결정계수(R-squared)라는 개념을 새로 다룹니다. OLS는 Ordinary Least Squares(최소제곱법)의 줄임말로, 실제값과 예측값의 차이(잔차)를 제곱해서 더한 값이 가장 작아지도록 회귀계수를 구하는 가장 기본적인 선형회귀 방법입니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기 및 요약 정보 확인**

In [12]:
import pandas as pd

df = pd.read_csv('crop_yield.csv')
print(df.shape)
df.head()

(260, 6)


,id,rainfall,avg_temp,fertilizer_amount,sunlight_hours,crop_yield
0,1,1073.6,23.7,183.4,9.3,6025.3
1,2,640.4,22.7,140.0,8.9,4746.9
2,3,988.9,17.7,174.8,7.1,4999.1
3,4,1023.4,18.1,146.5,9.2,4510.5
4,5,606.3,14.5,147.2,7.5,4146.9


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260 entries, 0 to 259
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 260 non-null    int64  
 1   rainfall           260 non-null    float64
 2   avg_temp           260 non-null    float64
 3   fertilizer_amount  260 non-null    float64
 4   sunlight_hours     260 non-null    float64
 5   crop_yield         260 non-null    float64
dtypes: float64(5), int64(1)
memory usage: 12.3 KB


260개 재배 구역의 데이터입니다. 모든 컬럼이 이미 숫자형이므로, 문제 1과 달리 범주형 변수를 숫자로 바꾸는 인코딩 과정은 필요하지 않습니다.

**셀 2) 다중선형 회귀모형 적합**

In [14]:
import statsmodels.api as sm

X = df.drop(columns=['id', 'crop_yield'])
y = df['crop_yield']
X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             crop_yield   R-squared:                       0.571
Model:                            OLS   Adj. R-squared:                  0.564
Method:                 Least Squares   F-statistic:                     84.90
Date:                Thu, 09 Jul 2026   Prob (F-statistic):           9.77e-46
Time:                        11:20:02   Log-Likelihood:                -1906.5
No. Observations:                 260   AIC:                             3823.
Df Residuals:                     255   BIC:                             3841.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const              1450.4761    237.77

로지스틱 회귀와 똑같이 `sm.add_constant()`로 절편을 추가하고, `sm.Logit` 대신 `sm.OLS`로 모형을 적합했습니다. `summary()` 표 맨 위의 `R-squared: 0.571`은 이 모형이 crop_yield의 변동 중 약 57.1%를 설명하고 있다는 뜻입니다. 아래쪽 계수표를 보면 rainfall, avg_temp, fertilizer_amount는 p-value가 0.000으로 뚜렷하게 유의하지만, sunlight_hours는 p-value가 0.273으로 0.05보다 훨씬 커서 유의하지 않습니다. 즉 일조 시간은 이 데이터 안에서는 수확량을 설명하는 데 통계적으로 의미 있는 역할을 하지 못하는 것으로 나타났습니다. 표 맨 아래(Omnibus, Durbin-Watson, Skew 등)는 잔차의 정규성이나 자기상관 같은 회귀모형의 추가 진단 통계량인데, 3유형 문제에서 직접 물어보는 경우는 드뭅니다.

**셀 3) 절편 제외 계수 합 (문제 2-1 정답)**

In [15]:
coef_sum = round(model.params.drop('const').sum(), 3)
print("2-1 정답:", coef_sum)

2-1 정답: 50.019


`model.params`에서 `.drop('const')`로 절편 행을 제외한 뒤, 남은 네 변수(rainfall, avg_temp, fertilizer_amount, sunlight_hours)의 계수를 모두 더했습니다. 문제 1-2에서는 "절편 포함"이었지만 이번 문제는 "절편을 제외한"이라고 명시되어 있으므로 정반대의 처리가 필요합니다. 이렇게 매 문제마다 절편의 포함 여부가 다르게 지시될 수 있으므로, 합계를 구하기 전에 지문을 다시 한번 확인하는 습관이 중요합니다.

**셀 4) 유의한 변수 선별**

In [16]:
sig_vars = model.pvalues[model.pvalues < 0.05].index.tolist()
print("유의한 변수(절편 포함):", sig_vars)

유의한 변수(절편 포함): ['const', 'rainfall', 'avg_temp', 'fertilizer_amount']


문제 1의 셀 6과 완전히 동일한 방식입니다. p-value가 0.05보다 작은 변수만 골라냈고, 그 결과 sunlight_hours가 제외된 4개(절편 포함)가 남았습니다.

**셀 5) 유의한 변수만으로 재적합**

In [17]:
X2 = X[sig_vars]
model2 = sm.OLS(y, X2).fit()
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:             crop_yield   R-squared:                       0.569
Model:                            OLS   Adj. R-squared:                  0.564
Method:                 Least Squares   F-statistic:                     112.7
Date:                Thu, 09 Jul 2026   Prob (F-statistic):           1.54e-46
Time:                        11:22:29   Log-Likelihood:                -1907.1
No. Observations:                 260   AIC:                             3822.
Df Residuals:                     256   BIC:                             3836.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const              1591.4186    200.24

sunlight_hours를 제거하고 재적합한 결과, 남은 세 변수 모두 p-value 0.000으로 뚜렷하게 유의합니다.

> 💡 재적합 후 R-squared가 원래 모형(0.571)보다 살짝 낮아진(0.569) 것을 이상하게 여길 수 있는데, 이는 자연스러운 현상입니다. R-squared는 변수를 추가하면 그 변수가 쓸모없더라도 무조건 조금이라도 오르는 성질이 있습니다. 반대로 변수를 제거하면 아주 미미하게라도 R-squared가 낮아질 수 있습니다. 대신 Adj. R-squared(수정된 결정계수, 변수 개수에 대한 불이익이 반영된 지표)는 두 모형에서 0.564로 동일하게 유지되었는데, 이는 sunlight_hours가 모형에 실질적인 설명력을 더하지 못하는 변수였다는 것을 보여주는 근거입니다.

**셀 6) R-squared 확인 (문제 2-2 정답)**

In [18]:
r2 = round(model2.rsquared, 3)
print("2-2 정답:", r2)

2-2 정답: 0.569


`model.rsquared`는 summary() 표에서 눈으로 확인했던 R-squared 값을 코드로 바로 꺼내는 속성입니다. summary() 전체를 출력해서 눈으로 찾는 대신, 필요한 값만 이렇게 속성으로 꺼내 쓰는 습관을 들이면 여러 소문항을 빠르고 정확하게 처리할 수 있습니다.

**셀 7) 새로운 재배 구역의 수확량 예측 (문제 2-3 정답)**

In [19]:
new_data = pd.DataFrame({'const': [1], 'rainfall': [1000], 'avg_temp': [24], 'fertilizer_amount': [200]})
print(new_data)

pred = model2.predict(new_data)
print("2-3 정답:", round(pred.values[0], 3))

   const  rainfall  avg_temp  fertilizer_amount
0      1      1000        24                200
2-3 정답: 4975.921


문제 1-3과 같은 방식으로, const를 포함해 모형에 사용된 컬럼과 정확히 같은 이름과 순서로 새 데이터를 구성했습니다. sunlight_hours는 2-2 모형에서 이미 제거되었으므로 새 데이터에도 포함하지 않았습니다.

### 전체 코드

In [20]:
import pandas as pd
import statsmodels.api as sm

df = pd.read_csv('crop_yield.csv')

X = df.drop(columns=['id', 'crop_yield'])
y = df['crop_yield']
X = sm.add_constant(X)

# 전체 변수로 적합
model = sm.OLS(y, X).fit()

# 2-1: 절편 제외 계수 합
coef_sum = round(model.params.drop('const').sum(), 3)
print("2-1 정답:", coef_sum)

# 유의한 변수만 재적합
sig_vars = model.pvalues[model.pvalues < 0.05].index.tolist()
X2 = X[sig_vars]
model2 = sm.OLS(y, X2).fit()

# 2-2: R-squared
r2 = round(model2.rsquared, 3)
print("2-2 정답:", r2)

# 2-3: 새 데이터 예측
new_data = pd.DataFrame({'const': [1], 'rainfall': [1000], 'avg_temp': [24], 'fertilizer_amount': [200]})
pred = round(model2.predict(new_data).values[0], 3)
print("2-3 정답:", pred)

2-1 정답: 50.019
2-2 정답: 0.569
2-3 정답: 4975.921


### 정리

문제 2는 문제 1과 같은 틀(전체 적합 → 유의변수 재적합 → 해석/예측)을 다중선형 회귀에 적용한 것이었지만, 세부적으로는 로지스틱 회귀와 구분되는 개념들을 다뤘습니다. `sm.Logit` 대신 `sm.OLS`를 사용했고, 오즈비 대신 R-squared라는 회귀모형 고유의 설명력 지표를 다뤘으며, 절편 포함 여부가 문제마다 다르게 지시될 수 있다는 것도 확인했습니다. 개인적으로는, 3유형에서 점수를 안정적으로 얻으려면 오즈비·R-squared 같은 통계 개념 자체를 이해하는 것도 중요하지만, 그에 못지않게 "절편 포함/제외", "몇 번째 소수 자리까지"처럼 문제마다 미묘하게 달라지는 지시사항을 놓치지 않는 꼼꼼함이 실수를 줄이는 데 더 크게 작용하는 것 같습니다.

---

## 이번 회차에서 사용한 핵심 개념 정리
| 상황 | 함수/개념 | 핵심 이유 |
|---|---|---|
| statsmodels 회귀분석에서 절편 포함시키기 | `sm.add_constant(X)` | statsmodels는 sklearn과 달리 절편을 자동으로 추가하지 않음 |
| 이진 종속변수에 대한 회귀 | `sm.Logit(y, X).fit()` | 로지스틱 회귀, 계수는 로그 오즈 단위 |
| 연속형 종속변수에 대한 회귀 | `sm.OLS(y, X).fit()` | 최소제곱법 기반의 선형회귀 |
| 로그 오즈를 오즈비로 변환 | `np.exp(계수)` | "몇 배 더 위험/유리한가"로 해석 가능해짐 |
| 유의한 변수만 골라내기 | `model.pvalues[model.pvalues < 0.05].index` | summary() 표를 눈으로 보지 않고 코드로 조건 처리 |
| 회귀계수만 따로 꺼내기 | `model.params` | 필요한 값만 빠르게 조회 |
| 모형의 설명력 확인 | `model.rsquared` | 종속변수 변동 중 모형이 설명하는 비율 |
| 새 데이터에 대한 예측 | `model.predict(new_data)` | 학습에 사용한 컬럼과 이름·순서를 정확히 맞춰야 함 (const 포함) |